# 🎙️ Qwen3-TTS-0.6B — Servidor TTS para AI Waifu VTuber
**Modelo correcto:** `Qwen/Qwen3-TTS-0.6B` (NO Qwen2.5-Omni)

Este notebook levanta un servidor Flask + ngrok que tu `ui.py` consumirá como **`Qwen TTS (Remote)`**.

### ✅ Pasos rápidos
1. En Kaggle: **Settings → Accelerator → GPU T4 x2** (o P100)
2. Activa **Internet** en Settings
3. Pega tu token de ngrok en la celda de configuración
4. Ejecuta todas las celdas en orden (`Run All`)
5. Copia la URL pública ngrok → pégala en `ui.py` → pestaña TTS → **Qwen TTS (Remote)** → campo **Remote URL**

## 🖥️ Celda 0 — Verificar GPU
Ejecuta esto primero para confirmar que tienes GPU asignada.

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else '❌ GPU no disponible — ve a Settings → Accelerator → GPU T4 x2')

import torch
print(f'\n✅ PyTorch: {torch.__version__}')
print(f'🎮 CUDA disponible: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'🖥️  GPU: {torch.cuda.get_device_name(0)}')
    print(f'💾 VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 📦 Celda 1 — Instalar dependencias

In [ ]:
%%bash
pip install -q --upgrade pip
pip install -q \
    transformers>=4.51.0 \
    accelerate \
    soundfile \
    scipy \
    flask \
    pyngrok \
    huggingface_hub

echo '✅ Dependencias instaladas'

## ⚙️ Celda 2 — Configuración (edita aquí)

Obtén tu token gratuito en: https://dashboard.ngrok.com/get-started/your-authtoken

In [ ]:
# ============================================================
#  🔧 EDITA ESTOS VALORES
# ============================================================
NGROK_TOKEN = "PEGA_TU_TOKEN_AQUI"   # ← tu authtoken de ngrok
MODEL_ID    = "Qwen/Qwen3-TTS-0.6B"  # ← modelo correcto (NO Qwen2.5-Omni)
PORT        = 5000
# ============================================================

print(f'Modelo  : {MODEL_ID}')
print(f'Puerto  : {PORT}')
print(f'ngrok   : {"configurado ✅" if NGROK_TOKEN != "PEGA_TU_TOKEN_AQUI" else "⚠️ NO configurado — pega tu token"}')

## 🤖 Celda 3 — Cargar modelo Qwen3-TTS-0.6B

In [ ]:
import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline

# Qwen3-TTS usa el pipeline de text-to-speech de transformers
device = "cuda" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

print(f'⏳ Cargando {MODEL_ID} en {device} ({torch_dtype})...')
print('   Primera vez: descarga ~1.3 GB, puede tardar 2-4 minutos...')

# Pipeline oficial de Qwen3-TTS
tts_pipe = pipeline(
    task="text-to-speech",
    model=MODEL_ID,
    device=device,
    torch_dtype=torch_dtype,
)

print(f'\n✅ Modelo {MODEL_ID} cargado correctamente')
print('🎤 Listo para generar audio')

## 🧪 Celda 4 — Test rápido del modelo (opcional)

In [ ]:
import soundfile as sf
import io
from IPython.display import Audio, display

test_text = "Hola, soy Miko Kawaii, tu VTuber favorita. Estoy lista para el stream!"

print(f'🎤 Generando audio de prueba: "{test_text[:50]}..."')
output = tts_pipe(test_text)

audio_array = output["audio"]
sample_rate = output["sampling_rate"]

# Guardar y reproducir
sf.write("/kaggle/working/test_output.wav", audio_array.T, sample_rate)
print(f'✅ Audio generado — sample rate: {sample_rate} Hz')
display(Audio("/kaggle/working/test_output.wav"))

## 🌐 Celda 5 — Servidor Flask + ngrok

Este servidor expone el endpoint `/tts` que consume `ui.py` con **Qwen TTS (Remote)**.

La URL pública aparecerá al ejecutar — **cópiala a ui.py**.

In [ ]:
import threading
import io
import soundfile as sf
import numpy as np
from flask import Flask, request, jsonify, send_file
from pyngrok import ngrok, conf

# ── Configurar ngrok ─────────────────────────────────────────
conf.get_default().auth_token = NGROK_TOKEN

# ── App Flask ────────────────────────────────────────────────
app = Flask(__name__)

@app.route('/health', methods=['GET'])
def health():
    """Endpoint de salud para verificar que el servidor está activo"""
    return jsonify({
        'status': 'ok',
        'model': MODEL_ID,
        'device': device
    })

@app.route('/tts', methods=['POST'])
def synthesize():
    """
    Endpoint principal — compatible con generate_qwen_remote_tts() de ui.py
    Recibe: JSON { "text": "..." }
    Devuelve: audio/wav
    """
    try:
        data = request.get_json(force=True)
        text = data.get('text', '').strip()

        if not text:
            return jsonify({'error': 'Campo "text" vacío o ausente'}), 400

        print(f'📝 Sintetizando: {text[:80]}...' if len(text) > 80 else f'📝 Sintetizando: {text}')

        # Generar audio con Qwen3-TTS-0.6B
        # ✅ SIN prompt de transcript — pronuncia cualquier idioma naturalmente
        output = tts_pipe(text)

        audio_array = output["audio"]   # shape: (channels, samples) o (samples,)
        sample_rate = output["sampling_rate"]

        # Asegurar mono / transponer si es necesario
        if audio_array.ndim == 2:
            audio_data = audio_array.T   # (samples, channels) para soundfile
        else:
            audio_data = audio_array

        # Escribir WAV en memoria
        buf = io.BytesIO()
        sf.write(buf, audio_data, sample_rate, format='WAV', subtype='PCM_16')
        buf.seek(0)

        size_kb = buf.getbuffer().nbytes / 1024
        print(f'✅ Audio generado — {size_kb:.1f} KB | {sample_rate} Hz')

        return send_file(
            buf,
            mimetype='audio/wav',
            as_attachment=False,
            download_name='output.wav'
        )

    except Exception as e:
        import traceback
        print(f'❌ Error: {e}')
        print(traceback.format_exc())
        return jsonify({'error': str(e)}), 500


# ── Lanzar servidor en hilo separado ─────────────────────────
def run_flask():
    app.run(host='0.0.0.0', port=PORT, debug=False, use_reloader=False)

flask_thread = threading.Thread(target=run_flask, daemon=True)
flask_thread.start()

# ── Abrir túnel ngrok ─────────────────────────────────────────
import time
time.sleep(1)  # esperar que Flask arranque

public_url = ngrok.connect(PORT).public_url

print('\n' + '='*60)
print('🚀 SERVIDOR LISTO')
print('='*60)
print(f'\n🌐 URL PÚBLICA NGROK:')
print(f'   {public_url}')
print(f'\n📋 PÉGALA EN UI.PY:')
print(f'   Pestaña TTS → "Qwen TTS (Remote)" → campo "Remote URL"')
print(f'   → Escribe: {public_url}')
print(f'\n🔗 Endpoints disponibles:')
print(f'   GET  {public_url}/health  ← verificar estado')
print(f'   POST {public_url}/tts     ← generar audio')
print('='*60)
print('\n⚠️  Mantén esta celda corriendo — si la detienes, se corta el servidor')

## 🔁 Celda 6 — Keep-alive (mantener sesión activa)

Kaggle desconecta sesiones inactivas. Este loop mantiene la GPU ocupada.

In [ ]:
import time

print('🔄 Keep-alive activo — el servidor seguirá corriendo')
print('   Presiona el botón ⏹ de la celda para detener')
print(f'   URL: {public_url}')

try:
    counter = 0
    while True:
        time.sleep(60)
        counter += 1
        print(f'   ⏱ [{counter} min] Servidor activo — {public_url}')
except KeyboardInterrupt:
    print('\n🛑 Keep-alive detenido')

---
## 📖 Guía completa

### 🎮 Cómo activar la GPU en Kaggle
1. Abre el notebook en Kaggle
2. Panel derecho → **Settings** (engranaje ⚙️)
3. **Accelerator** → selecciona **GPU T4 x2** (gratis, hasta 30h/semana)
4. Asegúrate que **Internet** esté en **ON**
5. Guarda y ejecuta

### 🔑 Dónde poner el ngrok en ui.py

En tu `ui.py` el proveedor **`Qwen TTS (Remote)`** ya tiene un campo de URL.

**Pasos exactos:**
1. Ejecuta este notebook en Kaggle hasta la Celda 5
2. Copia la URL que aparece: `https://xxxx.ngrok-free.app`
3. En `ui.py` → pestaña **TTS**
4. Selecciona proveedor: **`Qwen TTS (Remote)`**
5. Pega la URL en el campo **Remote URL**
6. Haz clic en **💾 Guardar configuración TTS**

El código de `ui.py` automáticamente llama a `POST {url}/tts` con `{"text": "..."}` — exactamente lo que este servidor espera.

### 🗣️ Por qué NO se pone transcript de audio de referencia
Qwen3-TTS-0.6B puede clonar voz pero cuando se le da un transcript en otro idioma, **pronuncia con acento del idioma del transcript**. Sin transcript, el modelo detecta el idioma del texto automáticamente y pronuncia de forma nativa.

### ⚠️ Notas importantes
- La sesión de Kaggle dura máximo **12 horas** por ejecución
- La URL de ngrok **cambia cada vez** que reinicias — actualiza en ui.py
- ngrok gratuito permite 1 túnel simultáneo y ~40 req/min
- El modelo Qwen3-TTS-0.6B ocupa ~1.3 GB de VRAM